# Proyecto integrador · Machine Learning I — Entrega final
**Prevalencia y predictores de diabetes en adultos (contexto latinoamericano)**  
**Jean Paul Rojas Rojas y Mariana Uribe Vanegas · Noviembre de 2026**

**Este cuaderno es autocontenido:** contiene todo el código de preprocesamiento, validación, ajuste, evaluación, interpretación y exportación del modelo. **No importa `modelado.py` ni necesita ejecutarlo.**

**Cómo usar:** coloca el CSV en `datos/base_diabetes_ensanut2025.csv` (o junto al notebook) y ejecuta **Run All** desde la carpeta del proyecto. Las tablas y figuras se guardarán en `resultados/` y el modelo en `modelos/modelo_diabetes.joblib` para `app.py`.

**Advertencias:** esta es una demostración académica, no un instrumento de diagnóstico. La procedencia exacta del CSV identificado como ENSANUT 2025 requiere verificación documental. La segunda entrega ya consultó datos de este mismo conjunto, por lo que esta nueva división por hogares **no equivale a validación externa ciega**. La IA generativa se utilizó como apoyo para estructurar y depurar código; el equipo debe comprenderlo y sustentarlo.

## 1. Bibliotecas, rutas y funciones auxiliares
Se configura la semilla, se declaran los 17 predictores y se encapsula el preprocesamiento dentro de `Pipeline` para prevenir fuga de información.

In [1]:
"""Proyecto integrador ML I - Jean Paul Rojas y Mariana Uribe.
Ejecutar desde la carpeta raíz: python modelado.py
Datos: archivo suministrado al equipo, fuente institucional pendiente de verificar.
"""
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_curve, precision_recall_curve)

BASE = Path.cwd()
DATOS = BASE / 'datos' / 'base_diabetes_ensanut2025.csv'
if not DATOS.exists():
    DATOS = BASE / 'base_diabetes_ensanut2025.csv'
if not DATOS.exists():
    raise FileNotFoundError('No se encuentra base_diabetes_ensanut2025.csv en datos/ ni en la carpeta del notebook')
OUT = BASE / 'resultados'; OUT.mkdir(exist_ok=True)
MODELOS = BASE / 'modelos'; MODELOS.mkdir(exist_ok=True)
SEED = 42
NUM = ['EDAD','IMC','PAS_MEDIA','PAD_MEDIA']
CAT = ['SEXO','TABAQUISMO','CONSUMO_ALCOHOL','NIVEL_EDUCATIVO','ESTRATO_URBANO']
BIN = ['HIPERTENSION','COLESTEROL_ALTO','TRIGLICERIDOS_ALTOS','ENF_RENAL',
       'ENF_CARDIOVASCULAR','DIF_CAMINAR','SIN_SEGURO_SALUD','TRABAJA']
FEATURES = NUM + CAT + BIN

def pipeline(modelo):
    prep = ColumnTransformer([
        ('num', Pipeline([('imputador', SimpleImputer(strategy='median')),
                          ('escalador', StandardScaler())]), NUM),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), CAT),
        ('bin', 'passthrough', BIN)
    ])
    return Pipeline([('preprocesamiento', prep), ('modelo', modelo)])

def metricas(y, proba, t):
    pred = (np.asarray(proba) >= t).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'umbral':float(t),'roc_auc':float(roc_auc_score(y,proba)),
            'pr_auc':float(average_precision_score(y,proba)),
            'sensibilidad':float(recall_score(y,pred,zero_division=0)),
            'especificidad':float(tn/(tn+fp)),
            'precision':float(precision_score(y,pred,zero_division=0)),
            'f1':float(f1_score(y,pred,zero_division=0)),
            'exactitud':float((tp+tn)/(tp+tn+fp+fn)),
            'VP':int(tp),'VN':int(tn),'FP':int(fp),'FN':int(fn)}

def umbral_youden(y, probs):
    # selección de umbral exclusivamente en probabilidades OOF de entrenamiento
    fpr, tpr, thresholds = roc_curve(y, probs)
    validos = np.isfinite(thresholds)
    ts=thresholds[validos]; js=(tpr-fpr)[validos]
    return float(ts[np.argmax(js)])

def bootstrap_ci(y, probs, t, n=500):
    rng=np.random.default_rng(SEED); y=np.asarray(y); probs=np.asarray(probs)
    vals=[]
    for _ in range(n):
        indices=rng.integers(0,len(y),len(y))
        yy=y[indices]
        if len(np.unique(yy))<2: continue
        mm=metricas(yy,probs[indices],t)
        vals.append([mm['sensibilidad'],mm['precision'],mm['roc_auc'],mm['pr_auc']])
    a=np.asarray(vals)
    return {k:[float(x) for x in np.quantile(a[:,j],[.025,.975])]
            for j,k in enumerate(['sensibilidad','precision','roc_auc','pr_auc'])}

## 2. Lectura, calidad mínima y datos de entrada
Comprobamos el número de observaciones, la unicidad del identificador individual y los valores de la etiqueta; `FOLIO_I` se usa únicamente para agrupar hogares y nunca como predictor.

In [2]:
df=pd.read_csv(DATOS)
assert len(df)==8475, 'Revisar versión del CSV'
assert df['FOLIO_INT'].is_unique, 'Hay individuos repetidos'
assert set(df['DIABETES'].unique())=={0,1}
X=df[FEATURES].copy(); y=df['DIABETES'].astype(int); groups=df['FOLIO_I']

## 3. Separación sin hogares compartidos
Se usa una partición agrupada aproximada 80/20. `StratifiedGroupKFold` evita que miembros del mismo hogar estén en entrenamiento y prueba; los tamaños pueden variar ligeramente.

In [3]:
# Holdout aproximado del 20%, separando hogares completos.
split_holdout=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
idx_train,idx_test=next(split_holdout.split(X,y,groups))
xtr,xte=X.iloc[idx_train],X.iloc[idx_test]
ytr,yte=y.iloc[idx_train],y.iloc[idx_test]
gtr,gte=groups.iloc[idx_train],groups.iloc[idx_test]
assert set(gtr).isdisjoint(set(gte)), 'Fuga entre hogares'
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
split_info={'n_total':len(df),'n_train':len(xtr),'n_test':len(xte),
            'hogares_train':gtr.nunique(),'hogares_test':gte.nunique(),
            'hogares_compartidos':len(set(gtr)&set(gte)),
            'prevalencia_train':float(ytr.mean()),'prevalencia_test':float(yte.mean()),
            'advertencia':'La prueba anterior fue consultada en fase 2; el holdout por hogar de fase 3 NO es completamente ciego porque hay reutilización del dataset y decisiones previas.'}
(OUT/'particion.json').write_text(json.dumps(split_info,indent=2,ensure_ascii=False),encoding='utf-8')
print('PARTICIÓN POR HOGAR:',split_info)

PARTICIÓN POR HOGAR: {'n_total': 8475, 'n_train': 6780, 'n_test': 1695, 'hogares_train': 5489, 'hogares_test': 1381, 'hogares_compartidos': 0, 'prevalencia_train': 0.16622418879056047, 'prevalencia_test': 0.1657817109144543, 'advertencia': 'La prueba anterior fue consultada en fase 2; el holdout por hogar de fase 3 NO es completamente ciego porque hay reutilización del dataset y decisiones previas.'}


## 4. Comparación de modelos con validación cruzada agrupada
Se comparan la línea base, regresión logística, QDA, Naive Bayes y KNN. Las cuadrículas de KNN y Naive Bayes se amplían según la observación de la segunda entrega. La selección se hace sin mirar la prueba final.

In [4]:
# Selección por ROC-AUC en entrenamiento agrupado. Regresion logística sigue siendo candidata principal.
modelos={
    'Mayoría': (DummyClassifier(strategy='most_frequent'),{}),
    'Logística L2': (LogisticRegression(C=.1,class_weight='balanced',max_iter=2000,random_state=SEED),{'modelo__C':[.01,.1,1,10]}),
    'QDA':(QuadraticDiscriminantAnalysis(),{'modelo__reg_param':[.1,.3,.5,.7,.9]}),
    'Naive Bayes':(GaussianNB(),{'modelo__var_smoothing':[.001,.01,.1,.3,.5,1.]}),
    'KNN':(KNeighborsClassifier(),{'modelo__n_neighbors':[21,41,61,81,101], 'modelo__weights':['uniform','distance'], 'modelo__metric':['manhattan','euclidean']})
}
filas=[]; mejores={}
for nombre,(est,grid) in modelos.items():
    print('Validando:',nombre,flush=True)
    pipe=pipeline(est)
    if grid:
        gs=GridSearchCV(pipe,grid,scoring='roc_auc',cv=cv,n_jobs=1,refit=True,error_score='raise')
        gs.fit(xtr,ytr,groups=gtr)
        pipe=gs.best_estimator_; params=gs.best_params_
    else:
        params={}
    res=cross_validate(pipe,xtr,ytr,groups=gtr,cv=cv,scoring={'auc':'roc_auc','recall':'recall','precision':'precision','f1':'f1'},n_jobs=1)
    filas.append({'modelo':nombre,'parametros':str(params),'auc_media':float(res['test_auc'].mean()),'auc_de':float(res['test_auc'].std()),'sensibilidad_media_050':float(res['test_recall'].mean()),'sensibilidad_de_050':float(res['test_recall'].std())})
    mejores[nombre]=pipe
comparacion=pd.DataFrame(filas).sort_values('auc_media',ascending=False)
comparacion.to_csv(OUT/'comparacion_cv_agrupada.csv',index=False)
print(comparacion.to_string(index=False))

Validando: Mayoría


Validando: Logística L2


/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/pyvenv/lib/python3.13/site-packages/sklearn/metrics/_

Validando: QDA


Validando: Naive Bayes


Validando: KNN


      modelo                                                                                parametros  auc_media   auc_de  sensibilidad_media_050  sensibilidad_de_050
         KNN {'modelo__metric': 'manhattan', 'modelo__n_neighbors': 81, 'modelo__weights': 'distance'}   0.802226 0.022106                0.047036             0.007796
         QDA                                                                {'modelo__reg_param': 0.1}   0.801256 0.021427                0.349644             0.032271
 Naive Bayes                                                            {'modelo__var_smoothing': 0.1}   0.795787 0.017898                0.464964             0.011137
Logística L2                                                                        {'modelo__C': 0.1}   0.793178 0.016399                0.725821             0.019068
     Mayoría                                                                                        {}   0.500000 0.000000                0.000000             0

## 5. Umbral definido con predicciones fuera de pliegue y evaluación final
La regresión logística L2 se conserva como modelo principal por interpretabilidad y sensibilidad. Se maximiza Youden con probabilidades OOF de entrenamiento y solo después se evalúa el conjunto de prueba agrupado. También se calculan intervalos bootstrap descriptivos.

In [5]:
# Se mantiene selección principal de L2 por simplicidad y explicabilidad, sin reelegir con TEST.
pipe=mejores['Logística L2']
probs_oof=cross_val_predict(pipe,xtr,ytr,groups=gtr,cv=cv,method='predict_proba',n_jobs=1)[:,1]
t=umbral_youden(ytr,probs_oof)
print('Umbral elegido exclusivamente en entrenamiento:',t)
pipe.fit(xtr,ytr)
probs=pipe.predict_proba(xte)[:,1]
pred=(probs>=t).astype(int)
resultado={'configuracion':str(pipe.named_steps['modelo'].get_params()),
           'umbral_oof':t,'youden_oof':metricas(ytr,probs_oof,t)['sensibilidad']+metricas(ytr,probs_oof,t)['especificidad']-1,
           'prueba_umbral_050':metricas(yte,probs,.5),
           'prueba_umbral_seleccionado':metricas(yte,probs,t),
           'ic95_bootstrap':bootstrap_ci(yte,probs,t)}
(OUT/'metricas_finales.json').write_text(json.dumps(resultado,ensure_ascii=False,indent=2),encoding='utf-8')
artefacto={'pipeline':pipe,'umbral':t,'features':FEATURES,'categorias':{c:sorted(df[c].dropna().unique().tolist()) for c in CAT},'version':'Fase3-hogares-2026'}
joblib.dump(artefacto, MODELOS/'modelo_diabetes.joblib')

Umbral elegido exclusivamente en entrenamiento: 0.3748248497914582


['/mnt/data/Entrega_Final_Notebook_ML1/modelos/modelo_diabetes.joblib']

## 6. Odds ratios y análisis de falsos negativos y positivos
Las odds ratios son asociaciones ajustadas del modelo **penalizado y ponderado**; no indican causalidad. En las variables numéricas corresponden a un incremento de una desviación estándar.

In [6]:
# OR: numéricas por +1 DE, binarias 0 a 1, categóricas vs referencia (alfabética).
pre=pipe.named_steps['preprocesamiento']
nombres=pre.get_feature_names_out()
coefs=pipe.named_steps['modelo'].coef_[0]
ors=pd.DataFrame({'variable':nombres,'beta':coefs,'odds_ratio':np.exp(coefs)})
ors.to_csv(OUT/'odds_ratios.csv',index=False)
err=xte.copy();err['observado']=yte.values;err['probabilidad']=probs;err['prediccion']=pred
err['tipo']=np.select([(err.observado==1)&(err.prediccion==0),(err.observado==0)&(err.prediccion==1),(err.observado==1)&(err.prediccion==1)],['FN','FP','VP'],default='VN')
analisis=err.groupby('tipo').agg(n=('observado','size'),edad_media=('EDAD','mean'),imc_medio=('IMC','mean'),hipertension_pct=('HIPERTENSION','mean'),probabilidad_media=('probabilidad','mean'))
analisis.to_csv(OUT/'perfil_errores.csv')

## 7. Sensibilidad por subgrupos
Se describe el comportamiento en sexo, estrato e hipertensión. Las diferencias observadas en subgrupos pequeños son inestables y no representan tasas poblacionales.

In [7]:
# Subgrupos explícitos: cuantos positivos y cuántos perdidos. Descriptivo, no causal.
sg=[]
for var in ['SEXO','ESTRATO_URBANO','HIPERTENSION']:
    for valor,dd in err.groupby(var,dropna=False):
        positives=dd[dd.observado==1]
        sg.append({'variable':var,'grupo':str(valor),'n_total':len(dd),'n_diabetes':len(positives),'sensibilidad':float((positives.prediccion==1).mean()) if len(positives) else None,'falsos_negativos':int((positives.prediccion==0).sum())})
pd.DataFrame(sg).to_csv(OUT/'desglose_subgrupos.csv',index=False)

## 8. Gráficas finales y matriz de confusión
Se muestran las curvas ROC y precisión-recall y la matriz de confusión para el umbral seleccionado. El modelo y los resultados quedan guardados y son consumidos por `app.py`.

In [8]:
fig,axs=plt.subplots(1,3,figsize=(15,4.3))
fpr,tpr,_=roc_curve(yte,probs);axs[0].plot(fpr,tpr,label=f"ROC-AUC {resultado['prueba_umbral_seleccionado']['roc_auc']:.3f}");axs[0].plot([0,1],[0,1],'--',color='grey');axs[0].set(xlabel='1 - especificidad',ylabel='Sensibilidad',title='ROC: prueba por hogar');axs[0].legend()
prec,rec,_=precision_recall_curve(yte,probs);axs[1].plot(rec,prec);axs[1].axhline(yte.mean(),linestyle='--',color='grey',label='Prevalencia');axs[1].set(xlabel='Sensibilidad',ylabel='Precisión',title='Curva precisión-sensibilidad');axs[1].legend()
cm=confusion_matrix(yte,pred);axs[2].imshow(cm,cmap='Blues');
for (i,j),v in np.ndenumerate(cm):axs[2].text(j,i,str(v),ha='center',va='center',fontsize=14)
axs[2].set(xticks=[0,1],yticks=[0,1],xlabel='Predicho',ylabel='Observado',title=f'Matriz de confusión, umbral={t:.3f}')
fig.tight_layout();fig.savefig(OUT/'figuras_evaluacion_final.png',dpi=180);plt.close(fig)
print('PRUEBA FINAL:',resultado['prueba_umbral_seleccionado'])
print('Archivos guardados en resultados/ y modelos/')

PRUEBA FINAL: {'umbral': 0.3748248497914582, 'roc_auc': 0.7669743842711673, 'pr_auc': 0.3473445792618792, 'sensibilidad': 0.8327402135231317, 'especificidad': 0.5912305516265912, 'precision': 0.2881773399014778, 'f1': 0.4281793229643184, 'exactitud': 0.6312684365781711, 'VP': 234, 'VN': 836, 'FP': 578, 'FN': 47}
Archivos guardados en resultados/ y modelos/


## 9. Archivos de salida y despliegue

El cuaderno genera automáticamente:

- `resultados/particion.json`, `comparacion_cv_agrupada.csv` y `metricas_finales.json`.
- `resultados/odds_ratios.csv`, `perfil_errores.csv` y `desglose_subgrupos.csv`.
- `resultados/figuras_evaluacion_final.png`.
- `modelos/modelo_diabetes.joblib`, que incluye **todo el Pipeline** y el umbral de clasificación.

La aplicación es un archivo independiente de Streamlit. **No hace falta ejecutar `modelado.py`.** Desde la terminal, en esta misma carpeta:

```bash
pip install -r requirements.txt
streamlit run app.py
```

Se recomienda ejecutar el notebook primero para regenerar el artefacto del modelo y los resultados. La aplicación presenta una puntuación predictiva académica, no una probabilidad clínica calibrada ni un diagnóstico.